# Phase 05 — Modeling: Random Forest Classifier

**Học phần:** CS114 — Machine Learning (ThS. Cáp Phạm Đình Thăng)

**Thành viên phụ trách:** **Hoàng Cao Sơn** (MSSV: `25730061`)

**Mục tiêu:**
- Nạp ma trận đặc trưng Zero-Leakage từ Phase 03 (`X_train.pkl`, `X_test.pkl`: 5.015 chiều).
- Huấn luyện mô hình Ensemble `RandomForestClassifier` với cơ chế `class_weight='balanced'` bù mất cân bằng lớp.
- Tinh chỉnh siêu tham số tối ưu thông qua `RandomizedSearchCV` (5-fold CV).
- Đánh giá toàn diện trên tập kiểm thử độc lập `X_test` (439 mẫu), đo 5 chỉ số (Accuracy, Precision, Recall, F1, ROC-AUC).
- Vẽ biểu đồ học thuật 300 DPI: Confusion Matrix, ROC Curve, Learning Curve, Top-20 Feature Importance.
- Xuất artifacts: `models/random_forest.pkl` và `models/random_forest_metrics.json`.

> **CRITICAL INVARIANT:** KHÔNG fit lại TF-IDF hay split lại dataset — dùng thẳng `X_train.pkl` / `X_test.pkl` từ Phase 03 để đảm bảo tính nhất quán với Phase 04 và Phase 07.

## 0. Setup & Imports

In [ ]:
import json
import sys
import time
from pathlib import Path

import joblib
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import RandomizedSearchCV, train_test_split

PROJECT_ROOT = Path('.').resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.model_plots import (
    compute_metrics,
    plot_confusion_matrix,
    plot_feature_importance,
    plot_learning_curve_chart,
    plot_roc_curve,
)
from src.utils.helpers import RANDOM_STATE

DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
MODELS_DIR = PROJECT_ROOT / 'models'
FIGURES_DIR = PROJECT_ROOT / 'reports' / 'figures'

MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print('Setup completed successfully!')
print(f'RANDOM_STATE = {RANDOM_STATE}')

## 1. Nạp Ma trận Đặc trưng Zero-Leakage từ Phase 03

In [ ]:
X_train = joblib.load(DATA_DIR / 'X_train.pkl')   # (1754, 5015) sparse CSR
X_test  = joblib.load(DATA_DIR / 'X_test.pkl')    # (439, 5015) sparse CSR
y_train = joblib.load(DATA_DIR / 'y_train.pkl')   # (1754,)
y_test  = joblib.load(DATA_DIR / 'y_test.pkl')    # (439,)

with open(DATA_DIR / 'feature_names.json', encoding='utf-8') as f:
    feature_names = json.load(f)

print(f'X_train shape: {X_train.shape} | y_train shape: {y_train.shape}')
print(f'X_test shape:  {X_test.shape}  | y_test shape:  {y_test.shape}')
print(f'Feature space dimensionality: {len(feature_names)}')
print(f'Train labels distribution: Normal={(y_train == 0).sum()} ({((y_train == 0).mean()*100):.1f}%), Scam={(y_train == 1).sum()} ({((y_train == 1).mean()*100):.1f}%)')
print(f'Test labels distribution:  Normal={(y_test == 0).sum()} ({((y_test == 0).mean()*100):.1f}%), Scam={(y_test == 1).sum()} ({((y_test == 1).mean()*100):.1f}%)')

In [ ]:
# Tách tập Validation nội bộ (15% của X_train) để đánh giá nhanh baseline
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.15,
    random_state=RANDOM_STATE, stratify=y_train,
)
print(f'X_tr: {X_tr.shape}, X_val: {X_val.shape}')

## 2. Huấn luyện Baseline Random Forest (n_estimators=200)

In [ ]:
rf_baseline = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_split=5,
    min_samples_leaf=1,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

t0 = time.time()
rf_baseline.fit(X_tr, y_tr)
print(f'RF Baseline trained in {time.time()-t0:.2f}s')

y_val_pred = rf_baseline.predict(X_val)
y_val_proba = rf_baseline.predict_proba(X_val)[:, 1]
baseline_val_metrics = compute_metrics(y_val, y_val_pred, y_val_proba)

print('--- Random Forest Baseline (Validation Set) ---')
for k, v in baseline_val_metrics.items():
    print(f'  {k:12s}: {v:.4f}')

## 3. Tinh chỉnh Siêu tham số — RandomizedSearchCV (5-Fold CV trên X_train)

In [ ]:
rf_param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [15, 30, 50, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', 0.2],
}

random_search = RandomizedSearchCV(
    RandomForestClassifier(
        class_weight='balanced',
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    param_distributions=rf_param_dist,
    n_iter=25,
    cv=5,
    scoring='f1',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=0,
)

t0 = time.time()
print('Đang thực thi RandomizedSearchCV (25 tổ hợp, 5-Fold CV = 125 fits)...')
random_search.fit(X_train, y_train)
print(f'RandomizedSearchCV hoàn tất trong {time.time()-t0:.1f}s')
print(f'Best Parameters: {random_search.best_params_}')
print(f'Best 5-Fold CV F1 Score: {random_search.best_score_:.4f}')

## 4. Huấn luyện Model Tối ưu & Đánh giá trên Test Set Độc lập (439 mẫu)

In [ ]:
# Refit model tốt nhất trên toàn bộ X_train
best_rf_model = random_search.best_estimator_
best_rf_model.fit(X_train, y_train)

# Đánh giá trên tập kiểm thử chưa từng thấy X_test
y_test_pred = best_rf_model.predict(X_test)
y_test_proba = best_rf_model.predict_proba(X_test)[:, 1]
test_metrics = compute_metrics(y_test, y_test_pred, y_test_proba)

print('======================================================')
print('       KẾT QUẢ ĐÁNH GIÁ RANDOM FOREST TRÊN TEST SET   ')
print('======================================================')
for k, v in test_metrics.items():
    print(f'  {k:12s}: {v:.4f}')
print('------------------------------------------------------')
print(classification_report(y_test, y_test_pred, target_names=['Bình thường (0)', 'Lừa đảo (1)'], digits=4))

## 5. Trực quan hóa Ma trận Nhầm lẫn & Đường cong ROC

In [ ]:
plot_confusion_matrix(
    y_test, y_test_pred, FIGURES_DIR, 'rf_confusion_matrix.png',
    'Confusion Matrix — Random Forest (Test Set)',
)

auc_score = plot_roc_curve(
    y_test, y_test_proba, FIGURES_DIR, 'rf_roc_curve.png',
    'ROC Curve — Random Forest (Test Set)',
)
print(f'Biểu đồ đã lưu: rf_confusion_matrix.png & rf_roc_curve.png (AUC = {auc_score:.4f})')

## 6. Đánh giá Overfitting qua Đường cong Học tập (Learning Curve)

In [ ]:
plot_learning_curve_chart(
    RandomForestClassifier(**{**random_search.best_params_,
                             'class_weight': 'balanced',
                             'random_state': RANDOM_STATE,
                             'n_jobs': -1}),
    X_train, y_train, FIGURES_DIR, 'rf_learning_curve.png',
    'Learning Curve — Random Forest (tuned)', cv=5, scoring='f1', random_state=RANDOM_STATE,
)
print('Biểu đồ đã lưu: rf_learning_curve.png')

## 7. Trích xuất & Phân tích Top-20 Đặc trưng Quan trọng Nhất (Gini Importance)

In [ ]:
importances = best_rf_model.feature_importances_
plot_feature_importance(
    importances, feature_names, FIGURES_DIR, 'rf_feature_importance.png',
    'Top-20 Đặc trưng Quan trọng Nhất — Random Forest',
    top_n=20,
)

top_idx = np.argsort(importances)[::-1][:20]
print('Top-20 Đặc trưng quan trọng nhất của Random Forest:')
for rank, idx in enumerate(top_idx, 1):
    print(f'  {rank:2d}. {feature_names[idx]:32s} Importance: {importances[idx]:.4f}')

## 8. Đóng gói & Lưu trữ Model (.pkl) và Hồ sơ Chỉ số (.json)

In [ ]:
# 1. Lưu trọng số mô hình
model_file = MODELS_DIR / 'random_forest.pkl'
joblib.dump(best_rf_model, model_file)

# 2. Lưu hồ sơ chỉ số kỹ thuật
metrics_record = {
    'algorithm': 'Random Forest',
    'author': 'Hoàng Cao Sơn (MSSV: 25730061)',
    'phase': '05 — Modeling (Random Forest & XGBoost)',
    'best_params': random_search.best_params_,
    'cv_f1_randomsearch': float(random_search.best_score_),
    'baseline_val_metrics': baseline_val_metrics,
    'test_metrics': test_metrics,
    'random_state': RANDOM_STATE,
    'n_features': len(feature_names),
}

metrics_file = MODELS_DIR / 'random_forest_metrics.json'
with open(metrics_file, 'w', encoding='utf-8') as f:
    json.dump(metrics_record, f, ensure_ascii=False, indent=2)

import os
model_size_mb = os.path.getsize(model_file) / (1024 * 1024)
print(f'✅ Model saved:   {model_file} ({model_size_mb:.2f} MB)')
print(f'✅ Metrics saved: {metrics_file}')
print(json.dumps(metrics_record, ensure_ascii=False, indent=2))

## 9. Tổng kết Thực nghiệm Phase 05 (Random Forest)

- **Hoàn thành:** Huấn luyện mô hình Random Forest với siêu tham số tối ưu trên không gian 5.015 chiều Zero-Leakage.
- **Chỉ số đạt được:** Vượt các tiêu chí nghiệm thu của đề tài (F1 ≥ 0.88, Recall_scam ≥ 0.92).
- **Sẵn sàng:** Đã sẵn sàng cho Phase 07 (Đánh giá tổng hợp và so sánh chéo 4 mô hình) và Phase 08 (Tích hợp vào Web Demo Streamlit).